In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

print(os.getcwd())
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, "pyproject.toml")) and project_root != os.path.dirname(project_root):
    project_root = os.path.dirname(project_root)
os.chdir(project_root)
print(os.getcwd())

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick 
import numpy as np
from model_utils.model_config import get_model_path
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name_list = ["qwen3-4b", "qwen3-8b", "qwen3-14b" ,"toolace-2.5-8b", "watt-tool-8b"]
# model_name_list = ["qwen3-4b", "qwen3-8b", "qwen3-14b" ,"toolace-2.5-8b", "watt-tool-8b"]
# model_name_list = ["qwen3-8b", "qwen3-14b" ,"toolace-2.5-8b"]

model_dsiplay_name_dict = {
    "qwen3-4b": "Qwen3-4B",
    "qwen3-8b": "Qwen3-8B",
    "qwen3-14b": "Qwen3-14B",
    "toolace-2.5-8b": "ToolACE-2.5-8B",
    "watt-tool-8b": "Watt-Tool-8B"
} 

add_param_num=1

In [ ]:
result = {}
for model_name in model_name_list:
    result[model_dsiplay_name_dict[model_name]] = {}

    file_dir = os.path.join("data", model_name)

    if model_name in ["toolace-2.5-8b", "watt-tool-8b"]:
        tokenizer = AutoTokenizer.from_pretrained(get_model_path(model_name))

    def check_tool_call(item):
        if model_name not in ["toolace-2.5-8b", "watt-tool-8b"]:
            return 1 if item["logits_info"]["tool_call_token_rank"]==0 else 0
        else:
            out_str = tokenizer.decode(item["logits_info"]["top_token_ids"][0], skip_special_tokens=True)
            if "]" not in out_str:
                if out_str.startswith("["):
                    if len(out_str) == 1:
                        return 1
                    elif out_str[1] == item["tool_name"][0]:
                        return 1
            return 0

    pair_file_path = os.path.join(file_dir, f"pair_add_{add_param_num}.json")
    with open(pair_file_path, "r", encoding="utf-8") as f:
        pair_list = json.load(f)
    pair_list = [check_tool_call(item) for item in pair_list]

    counterfactual_file_path = os.path.join(file_dir, f"counterfactual_add_{add_param_num}.json")
    with open(counterfactual_file_path, "r", encoding="utf-8") as f:
        counterfactual_list = json.load(f)
    counterfactual_list = [check_tool_call(item) for item in counterfactual_list]

    cf_add_file_path = os.path.join(file_dir, f"param_addition_add_{add_param_num}_tool_3.json")
    with open(cf_add_file_path, "r", encoding="utf-8") as f:
        cf_add_list = json.load(f)
    cf_add_list = [check_tool_call(item) for item in cf_add_list]

    cf_remove_file_path = os.path.join(file_dir, f"param_removal_add_{add_param_num}_tool_0.json")
    with open(cf_remove_file_path, "r", encoding="utf-8") as f:
        cf_remove_list = json.load(f)
    cf_remove_list = [check_tool_call(item) for item in cf_remove_list]

    result[model_dsiplay_name_dict[model_name]]["Original"] = pair_list
    result[model_dsiplay_name_dict[model_name]]["CF (Remove)"] = cf_remove_list
    result[model_dsiplay_name_dict[model_name]]["CF (Add)"] = cf_add_list
    result[model_dsiplay_name_dict[model_name]]["CF (Substitute)"] = counterfactual_list


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np

def plot_final_paper_version(result_data):
    """
    Bar chart for 4 conditions (Original, CF Add, CF Remove, CF Substitute).
    
    
    
    
    
    """
    
    # Global font setup
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.linewidth'] = 1.2
    
    # Bar configuration
    
    plot_configs = [
        {
            "key": "Original", 
            "label": "Original", 
            "color": "#CFEAF1"  # 
        },
        {
            "key": "CF (Add)", 
            "label": "CF (Add)", 
            "color": "#A1A9D0"  # 
        },
        {
            "key": "CF (Remove)", 
            "label": "CF (Remove)", 
            "color": "#F6CAE5"  # 
        },
        {
            "key": "CF (Substitute)", 
            "label": "CF (Substitute)", 
            "color": "#96CCCB"  # 
        }
    ]

    models = list(result_data.keys())
    x = np.arange(len(models))
    
    
    total_width = 0.8
    n_bars = len(plot_configs)
    bar_width = total_width / n_bars 
    
    
    data_map = {cfg["key"]: {"means": [], "errs": []} for cfg in plot_configs}
    max_height = 0 
    
    # Aggregate means/errs per bar
    for model in models:
        for cfg in plot_configs:
            key = cfg["key"]
            
            d_list = result_data[model].get(key, [])
            
            if len(d_list) > 0:
                val = np.mean(d_list) * 100
                
                err = (np.std(d_list, ddof=1) / np.sqrt(len(d_list))) * 100
            else:
                val, err = 0, 0
            
            data_map[key]["means"].append(val)
            data_map[key]["errs"].append(err)
            
            
            if (val + err) > max_height:
                max_height = val + err

    # Plot
    fig, ax = plt.subplots(figsize=(6, 3), dpi=300) 
    
    
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            
            
            if 0 < height < 0.01:
                label_text = '<0.01'
            elif height == 0:
                label_text = '0.00'
            else:
                label_text = f'{height:.2f}'
            
            
            offset = max_height * 0.02
            y_pos = height + offset if height > 0 else offset
            
            
            ax.annotate(label_text,
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=5.2, rotation=0, fontweight='bold') # 

    
    
    
    
    for i, cfg in enumerate(plot_configs):
        key = cfg["key"]
        means = data_map[key]["means"]
        errs = data_map[key]["errs"]
        
        
        offset = (i - (n_bars - 1) / 2) * bar_width
        
        rects = ax.bar(x + offset, means, bar_width,
                       label=cfg["label"],
                       color=cfg["color"],
                       edgecolor='black',
                       linewidth=1,
                    #    hatch='////',
                       capsize=0) 
        
        autolabel(rects)

    # Axes styling
    
    top_limit = max_height * 1.12 if max_height > 0 else 10 
    if top_limit > 100: top_limit = 110
    ax.set_ylim(0, top_limit)

    ax.yaxis.set_major_locator(MultipleLocator(10))
    
    
    ax.set_ylabel('Tool Invocation Rate (%)', fontsize=12.5, fontweight='bold')
    
    
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15, fontsize=12.5)
    
    
    leg = ax.legend(
        loc='upper right',
        bbox_to_anchor=(0.99, 0.98),   # 
        frameon=True,
        fontsize=8,
        ncol=1,
        borderaxespad=0.2
    )

    
    frame = leg.get_frame()
    frame.set_facecolor('white')
    frame.set_edgecolor('black')
    frame.set_linewidth(1.0)
    frame.set_alpha(1.0) 
    
    
    ax.yaxis.grid(True, linestyle='--', which='major', color='grey', alpha=0.3)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig("figs/behavior/performance_cf.pdf", bbox_inches="tight")
    plt.show()


plot_final_paper_version(result)